In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv("postings.csv")

print("Initial shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
df.head(2)

Initial shape: (123849, 31)
Columns:
['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0


In [3]:
dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)

df = df.drop_duplicates()

print("Shape after duplicate removal:", df.shape)

Duplicate rows: 0
Shape after duplicate removal: (123849, 31)


In [4]:
required_cols = ["title", "description", "formatted_experience_level"]

print("Missing values before cleaning:")
print(df[required_cols].isnull().sum())

df = df.dropna(subset=required_cols)

print("Shape after dropping required missing values:", df.shape)

Missing values before cleaning:
title                             0
description                       7
formatted_experience_level    29409
dtype: int64
Shape after dropping required missing values: (94440, 31)


In [5]:
for col in required_cols:
    df = df[df[col].astype(str).str.strip() != ""]

print("Shape after removing empty-string rows:", df.shape)

Shape after removing empty-string rows: (94440, 31)


In [6]:
df["desc_word_count"] = df["description"].astype(str).str.split().str.len()

print(df["desc_word_count"].describe())

count    94440.000000
mean       539.179903
std        300.557325
min          1.000000
25%        320.000000
50%        494.000000
75%        709.000000
max       3400.000000
Name: desc_word_count, dtype: float64


In [7]:
df = df[(df["desc_word_count"] >= 400) & (df["desc_word_count"] <= 700)]

print("Shape after description-length filtering:", df.shape)

Shape after description-length filtering: (36256, 32)


In [8]:
def map_experience_level(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()

    if any(term in x for term in ["intern", "entry", "junior", "associate", "trainee"]):
        return "junior"
    elif any(term in x for term in ["mid", "intermediate"]):
        return "mid"
    elif any(term in x for term in ["senior", "lead", "manager", "director", "executive", "principal", "head", "vp"]):
        return "senior"
    else:
        return np.nan

df["exp_level_3"] = df["formatted_experience_level"].apply(map_experience_level)

print(df["formatted_experience_level"].value_counts(dropna=False).head(20))
print("\nMapped labels:")
print(df["exp_level_3"].value_counts(dropna=False))

formatted_experience_level
Entry level         15875
Mid-Senior level    15104
Associate            2959
Director             1236
Internship            676
Executive             406
Name: count, dtype: int64

Mapped labels:
exp_level_3
junior    19510
mid       15104
senior     1642
Name: count, dtype: int64


In [9]:
salary_col = None
if "normalized_salary" in df.columns:
    salary_col = "normalized_salary"
elif "normalized_salery" in df.columns:
    salary_col = "normalized_salery"

def salary_signal(x):
    if pd.isna(x):
        return "unknown"
    try:
        x = float(x)
    except:
        return "unknown"

    if x < 50000:
        return "junior"
    elif x <= 100000:
        return "mid"
    else:
        return "senior"

if salary_col:
    df["salary_signal"] = df[salary_col].apply(salary_signal)
else:
    df["salary_signal"] = "unknown"

print(df["salary_signal"].value_counts(dropna=False))

salary_signal
unknown    26993
mid         3536
senior      3055
junior      2672
Name: count, dtype: int64


In [10]:
def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9,+/#.\-\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

if "skills_desc" in df.columns:
    df["skills_desc_clean"] = df["skills_desc"].apply(clean_text)
else:
    df["skills_desc_clean"] = np.nan

df[["skills_desc", "skills_desc_clean"]].head(3)

,skills_desc,skills_desc_clean
84,NaN,NaN
129,NaN,NaN
132,NaN,NaN


In [11]:
def count_skills(text):
    if pd.isna(text) or not str(text).strip():
        return 0
    parts = re.split(r",|;|\||\n", str(text))
    parts = [p.strip() for p in parts if p.strip()]
    parts = list(dict.fromkeys(parts))
    return len(parts)

df["skill_count"] = df["skills_desc_clean"].apply(count_skills)

def skill_count_band(n):
    if n == 0:
        return "unknown"
    elif n <= 3:
        return "low"
    elif n <= 7:
        return "medium"
    else:
        return "high"

df["skill_count_band"] = df["skill_count"].apply(skill_count_band)

print(df["skill_count"].describe())
print(df["skill_count_band"].value_counts(dropna=False))

count    36256.000000
mean         0.043331
std          0.777439
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         80.000000
Name: skill_count, dtype: float64
skill_count_band
unknown    35560
low          614
medium        47
high          35
Name: count, dtype: int64


In [12]:
advanced_skills = [
    "machine learning", "deep learning", "nlp", "computer vision",
    "aws", "azure", "gcp", "docker", "kubernetes", "terraform",
    "spark", "hadoop", "airflow", "pytorch", "tensorflow",
    "mlops", "big data", "data engineering", "architecture",
    "microservices", "llm", "genai"
]

mid_skills = [
    "python", "sql", "java", "excel", "tableau", "power bi",
    "statistics", "data analysis", "git", "linux", "etl",
    "scikit-learn", "pandas", "numpy", "matplotlib"
]

def keyword_count(text, keywords):
    if pd.isna(text):
        return 0
    text = str(text).lower()
    return sum(1 for kw in keywords if kw in text)

df["advanced_skill_count"] = df["skills_desc_clean"].apply(lambda x: keyword_count(x, advanced_skills))
df["mid_skill_count"] = df["skills_desc_clean"].apply(lambda x: keyword_count(x, mid_skills))

df[["advanced_skill_count", "mid_skill_count"]].describe()

,advanced_skill_count,mid_skill_count
count,36256.000000,36256.000000
mean,0.000303,0.001103
std,0.018934,0.039286
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,2.000000,3.000000


In [13]:
def skill_signal(row):
    adv = row["advanced_skill_count"]
    midc = row["mid_skill_count"]
    total = row["skill_count"]

    if adv >= 2 or total >= 8:
        return "senior"
    elif adv == 1 or midc >= 2 or 4 <= total <= 7:
        return "mid"
    elif 1 <= total <= 3:
        return "junior"
    else:
        return "unknown"

df["skill_signal"] = df.apply(skill_signal, axis=1)

signal_map = {"unknown": 0, "junior": 1, "mid": 2, "senior": 3}
df["skill_signal_num"] = df["skill_signal"].map(signal_map)

print(df["skill_signal"].value_counts(dropna=False))

skill_signal
unknown    35560
junior       611
mid           50
senior        35
Name: count, dtype: int64


In [14]:
def title_signal(title):
    title = str(title).lower()

    if any(w in title for w in ["intern", "trainee", "junior", "associate"]):
        return "junior"
    elif any(w in title for w in ["senior", "lead", "principal", "manager", "director", "head"]):
        return "senior"
    else:
        return "mid"

df["title_signal"] = df["title"].apply(title_signal)
df["title_signal_num"] = df["title_signal"].map(signal_map)

print(df["title_signal"].value_counts(dropna=False))

title_signal
mid       23383
senior     9845
junior     3028
Name: count, dtype: int64


In [15]:
def extract_years(text):
    text = str(text).lower()
    matches = re.findall(r"(\d+)\+?\s*(?:years|year|yrs|yr)", text)
    if matches:
        return max(int(m) for m in matches)
    return 0

df["years_experience"] = df["description"].apply(extract_years)

def years_signal(y):
    if y == 0:
        return "unknown"
    elif y <= 2:
        return "junior"
    elif y <= 5:
        return "mid"
    else:
        return "senior"

df["years_signal"] = df["years_experience"].apply(years_signal)
df["years_signal_num"] = df["years_signal"].map(signal_map)

print(df["years_signal"].value_counts(dropna=False))

years_signal
unknown    15422
senior      9707
mid         7300
junior      3827
Name: count, dtype: int64


In [16]:
final_cols = [
    "company_name",
    "title",
    "description",
    "location",
    "formatted_work_type",
    "formatted_experience_level",
    "skills_desc",
    "normalized_salary",
    "exp_level_3",
    "salary_signal",
    "skills_desc_clean",
    "skill_count",
    "skill_count_band",
    "advanced_skill_count",
    "mid_skill_count",
    "skill_signal",
    "skill_signal_num",
    "title_signal",
    "title_signal_num",
    "years_experience",
    "years_signal",
    "years_signal_num"
]

final_cols = [c for c in final_cols if c in df.columns]
df_final = df[final_cols].copy()

print("Final shape:", df_final.shape)
print(df_final.columns.tolist())
df_final.head(2)

Final shape: (36256, 22)
['company_name', 'title', 'description', 'location', 'formatted_work_type', 'formatted_experience_level', 'skills_desc', 'normalized_salary', 'exp_level_3', 'salary_signal', 'skills_desc_clean', 'skill_count', 'skill_count_band', 'advanced_skill_count', 'mid_skill_count', 'skill_signal', 'skill_signal_num', 'title_signal', 'title_signal_num', 'years_experience', 'years_signal', 'years_signal_num']


,company_name,title,description,location,formatted_work_type,formatted_experience_level,skills_desc,normalized_salary,exp_level_3,salary_signal,...,skill_count_band,advanced_skill_count,mid_skill_count,skill_signal,skill_signal_num,title_signal,title_signal_num,years_experience,years_signal,years_signal_num
84,Galerie Candy and Gifts,Quality Assurance Manager,Galerie is seeking an experienced Quality Assu...,"Hebron, KY",Full-time,Mid-Senior level,NaN,NaN,mid,unknown,...,unknown,0,0,unknown,0,senior,3,0,unknown,0
129,Webologix Ltd/ INC,Anaplan Developer,Job Title: Anaplan Developer\n\nLocations: US ...,United States,Full-time,Mid-Senior level,NaN,NaN,mid,unknown,...,unknown,0,0,unknown,0,mid,2,0,unknown,0


In [17]:
df_final.to_csv("linkedin_job_postings_after_eda_and_feature_engineering.csv", index=False)

print("Saved: linkedin_job_postings_after_eda_and_feature_engineering.csv")
print("Final rows:", len(df_final))

Saved: linkedin_job_postings_after_eda_and_feature_engineering.csv
Final rows: 36256


In [18]:
old_df = pd.read_csv("linkedin_job_postings_after_eda.csv")
new_df = pd.read_csv("linkedin_job_postings_after_eda_recreated.csv")

print("Old shape:", old_df.shape)
print("New shape:", new_df.shape)
print("Same columns:", list(old_df.columns) == list(new_df.columns))

Old shape: (36256, 22)
New shape: (36256, 22)
Same columns: True
